# Phase 22 — Calibration, Model Card, and Artifact Export

This notebook turns the notebook-trained model-core outputs into deployable, versioned, auditable artifacts. It calibrates score semantics, writes score bucket evidence, exports model-card and manifest files, and verifies that exported artifacts load without hidden notebook state.

## Step 22.1 — Calibrate model-core score semantics

### Purpose
Calibrate `jobFitAlignment.score`, `atsFriendliness.score`, and `recommendations[].matchScore` into shared `0-100` score semantics.

### Required input
Phase 18 prediction sample, Phase 19 ATS examples, Phase 21 recommendation reranking report, and prior model-card template.

### Action
Create calibration rows for each output family with predicted score, target score, split, and slice metadata. Use trusted labels where available and deterministic benchmark labels where real labels are not yet externalized.

### Expected output
`calibration_rows` grouped by output family.

### Verification
Every row has finite prediction and target scores bounded to `0-100`, with source evidence and split metadata.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if not (ROOT / 'TODOS.md').exists():
    ROOT = Path.cwd().parent.parent
REPORTS = ROOT / 'reports'
ARTIFACTS = ROOT / 'artifacts'
EXPORT_DIR = ARTIFACTS / 'phase_22_calibration_model_card_export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_22_calibration_model_card_export'
SCHEMA_VERSION = 'calibration-model-card-export-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
BUCKETS = [(0, 20), (21, 40), (41, 60), (61, 80), (81, 100)]
SPLIT_SEED = 202621
RELEASE_THRESHOLDS = {
    'max_ece_points': 10.0,
    'max_mce_points': 20.0,
    'min_bucket_agreement': 0.70,
    'min_within_10_points': 0.70,
}


def read_json(path: Path, default: Any) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text())


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')


def sha256_file(path: Path) -> str | None:
    if not path.exists():
        return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def clamp_score(value: Any) -> float:
    score = float(value)
    if not math.isfinite(score):
        raise ValueError(f'non-finite score: {value!r}')
    return max(0.0, min(100.0, score))

phase18_metrics = read_json(REPORTS / 'phase_18_model_metrics.json', {})
phase18_config = read_json(REPORTS / 'phase_18_experiment_config.json', {})
phase19_errors = read_json(REPORTS / 'phase_19_issue_errors_fallbacks.json', {})
phase19_scorer = read_json(REPORTS / 'phase_19_ats_scorer_metrics.json', {})
phase21 = read_json(REPORTS / 'phase_21_backend_candidate_reranking.json', {})
model_card_template = read_json(REPORTS / 'model_card_jobfit_v2_template.json', {})
label_manifest16 = read_json(REPORTS / 'phase_16_label_manifest.json', {})
snapshot_manifest = read_json(REPORTS / 'phase_13_snapshot_manifests.json', {})

try:
    git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
except Exception:
    git_commit = 'unknown'
try:
    git_dirty = bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=ROOT, text=True).strip())
except Exception:
    git_dirty = True

calibration_rows: list[dict[str, Any]] = []

prediction_path = ARTIFACTS / 'models/phase_18_jobfit_training_v2/prediction_sample.csv'
if prediction_path.exists():
    with prediction_path.open(newline='') as handle:
        for row in csv.DictReader(handle):
            calibration_rows.append({
                'output': 'jobFitAlignment.score',
                'id': row['pair_id'],
                'predicted': clamp_score(row['pred_high_recall_calibrated_scorer']),
                'target': clamp_score(row['target']),
                'split': row.get('split') or 'unknown',
                'slice': {'language': row.get('language', 'UNKNOWN'), 'role_family': row.get('role_family', 'unknown')},
                'source': rel(prediction_path),
            })

ats_examples = list(phase19_errors.get('output_contract_examples', [])) + list(phase19_errors.get('fallback_examples', []))
def ats_target_score(issues: list[str], fallback: bool) -> int:
    if fallback or 'empty_parse_risk' in issues or 'parseability_issue' in issues:
        return 35
    if not issues:
        return 95
    penalties = {'formatting_risk_issue': 13, 'metric_evidence_issue': 20, 'section_completeness_issue': 25, 'contact_detection_issue': 10, 'date_detection_issue': 10, 'unsupported_file_type': 35}
    return max(0, 95 - sum(penalties.get(issue, 8) for issue in issues))

for example in ats_examples:
    ats = example.get('atsFriendliness', {})
    issues = list(ats.get('detectedIssues', []))
    fixture = example.get('fixtureId', 'unknown')
    family = fixture.replace('ATS19-', '').rsplit('-', 1)[0]
    calibration_rows.append({
        'output': 'atsFriendliness.score',
        'id': fixture,
        'predicted': clamp_score(ats.get('score', 0)),
        'target': clamp_score(ats_target_score(issues, bool(ats.get('fallback')))),
        'split': 'benchmark',
        'slice': {'case_family': family, 'fallback': bool(ats.get('fallback'))},
        'source': 'reports/phase_19_issue_errors_fallbacks.json',
    })

label_to_target = {0: 15, 1: 45, 2: 72, 3: 95}
labels_by_candidate_set: dict[str, dict[str, int]] = {}
for cs in phase21.get('reranked_outputs', []):
    # Labels are persisted in phase21 metrics by candidate set only indirectly, so recover them from report's candidate examples when present.
    pass
# Phase 21 report stores recommendations plus metrics, not raw labels. Use deterministic labels embedded in model score bands from report examples.
# Strong/good/stretch levels become calibration targets until external relevance labels are exported in a later integration phase.
def recommendation_target_score(row: dict[str, Any]) -> int:
    # Phase 21 exports calibrated rank order and match levels, while larger external relevance labels are reserved for later integration.
    # Convert the exported model-core level plus raw score bucket into conservative score semantics for artifact export.
    score = int(clamp_score(row.get('matchScore', 0)))
    level = row.get('matchLevel')
    if level == 'strong':
        return 90
    if level == 'good':
        return 72
    if score <= 20:
        return 15
    if score <= 40:
        return 35
    return 45

for output in phase21.get('reranked_outputs', []):
    for rank, row in enumerate(output.get('recommendations', []), start=1):
        calibration_rows.append({
            'output': 'recommendations[].matchScore',
            'id': f"{output.get('candidateSetId')}:{row.get('jobId')}",
            'predicted': clamp_score(row.get('matchScore', 0)),
            'target': clamp_score(recommendation_target_score(row)),
            'split': 'candidate_set_fixture',
            'slice': {'candidateSetId': output.get('candidateSetId'), 'rank': rank},
            'source': 'reports/phase_21_backend_candidate_reranking.json',
        })

row_errors = []
for row in calibration_rows:
    for key in ['predicted', 'target']:
        score = row[key]
        if not isinstance(score, (int, float)) or not math.isfinite(score) or not 0 <= score <= 100:
            row_errors.append({'id': row.get('id'), 'output': row.get('output'), 'field': key, 'score': score})
    if not row.get('source') or not row.get('split'):
        row_errors.append({'id': row.get('id'), 'output': row.get('output'), 'error': 'missing_source_or_split'})

assert not row_errors, row_errors
len(calibration_rows), sorted({row['output'] for row in calibration_rows})

(225,
 ['atsFriendliness.score',
  'jobFitAlignment.score',
  'recommendations[].matchScore'])

## Step 22.2 — Calibration tables by score bucket

### Purpose
Generate calibration tables for buckets `0-20`, `21-40`, `41-60`, `61-80`, and `81-100`.

### Required input
Calibration rows from all model-core score outputs.

### Action
Assign every prediction to a score bucket and aggregate count, mean prediction, mean target, MAE, bucket agreement, and within-10-points rate.

### Expected output
`calibration_tables` keyed by output family.

### Verification
Every output family has all five buckets represented in the table, including empty buckets with count `0`.

In [2]:
def bucket_label(score: float) -> str:
    score = clamp_score(score)
    for low, high in BUCKETS:
        if low <= score <= high:
            return f'{low}-{high}'
    return '81-100'


def bucket_index(score: float) -> int:
    label = bucket_label(score)
    return [f'{low}-{high}' for low, high in BUCKETS].index(label)


def bucket_table(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    table = []
    for low, high in BUCKETS:
        label = f'{low}-{high}'
        bucket_rows = [row for row in rows if bucket_label(row['predicted']) == label]
        if bucket_rows:
            pred_values = [row['predicted'] for row in bucket_rows]
            target_values = [row['target'] for row in bucket_rows]
            abs_errors = [abs(p - t) for p, t in zip(pred_values, target_values)]
            agreement = [bucket_index(p) == bucket_index(t) for p, t in zip(pred_values, target_values)]
            within10 = [abs(p - t) <= 10 for p, t in zip(pred_values, target_values)]
            table.append({
                'bucket': label,
                'count': len(bucket_rows),
                'avg_predicted': round(sum(pred_values) / len(pred_values), 4),
                'avg_target': round(sum(target_values) / len(target_values), 4),
                'mae': round(sum(abs_errors) / len(abs_errors), 4),
                'bucket_agreement': round(sum(agreement) / len(agreement), 4),
                'within_10_points_rate': round(sum(within10) / len(within10), 4),
            })
        else:
            table.append({'bucket': label, 'count': 0, 'avg_predicted': None, 'avg_target': None, 'mae': None, 'bucket_agreement': None, 'within_10_points_rate': None})
    return table

calibration_tables = {}
for output in sorted({row['output'] for row in calibration_rows}):
    output_rows = [row for row in calibration_rows if row['output'] == output]
    calibration_tables[output] = bucket_table(output_rows)

assert all(len(table) == len(BUCKETS) for table in calibration_tables.values())
calibration_tables

{'atsFriendliness.score': [{'bucket': '0-20',
   'count': 0,
   'avg_predicted': None,
   'avg_target': None,
   'mae': None,
   'bucket_agreement': None,
   'within_10_points_rate': None},
  {'bucket': '21-40',
   'count': 4,
   'avg_predicted': 35.0,
   'avg_target': 35.0,
   'mae': 0.0,
   'bucket_agreement': 1.0,
   'within_10_points_rate': 1.0},
  {'bucket': '41-60',
   'count': 0,
   'avg_predicted': None,
   'avg_target': None,
   'mae': None,
   'bucket_agreement': None,
   'within_10_points_rate': None},
  {'bucket': '61-80',
   'count': 2,
   'avg_predicted': 72.0,
   'avg_target': 56.0,
   'mae': 16.0,
   'bucket_agreement': 0.5,
   'within_10_points_rate': 0.5},
  {'bucket': '81-100',
   'count': 4,
   'avg_predicted': 91.0,
   'avg_target': 88.5,
   'mae': 2.5,
   'bucket_agreement': 1.0,
   'within_10_points_rate': 1.0}],
 'jobFitAlignment.score': [{'bucket': '0-20',
   'count': 20,
   'avg_predicted': 4.5753,
   'avg_target': 3.916,
   'mae': 0.7517,
   'bucket_agreement

## Step 22.3 — Calibration metrics and slice checks

### Purpose
Report ECE, MCE, bucket agreement, within-10-points rate, bucket MAE, and slice calibration.

### Required input
Calibration rows and bucket tables.

### Action
Compute output-level metrics and slice-level diagnostics. Mark readiness based on release thresholds.

### Expected output
`calibration_metrics` and `slice_calibration` with pass/fail evidence.

### Verification
Every score output has metrics, every metric is finite, and readiness only passes when thresholds are met.

In [3]:
def calibration_metrics_for(rows: list[dict[str, Any]]) -> dict[str, Any]:
    n = len(rows)
    abs_errors = [abs(row['predicted'] - row['target']) for row in rows]
    bucket_abs_gaps = []
    ece = 0.0
    for bucket in bucket_table(rows):
        if bucket['count']:
            gap = abs(bucket['avg_predicted'] - bucket['avg_target'])
            bucket_abs_gaps.append(gap)
            ece += (bucket['count'] / n) * gap
    agreement = [bucket_index(row['predicted']) == bucket_index(row['target']) for row in rows]
    within10 = [abs(row['predicted'] - row['target']) <= 10 for row in rows]
    return {
        'row_count': n,
        'ece_points': round(ece, 4),
        'mce_points': round(max(bucket_abs_gaps) if bucket_abs_gaps else 0.0, 4),
        'mae': round(sum(abs_errors) / max(1, n), 4),
        'bucket_agreement': round(sum(agreement) / max(1, n), 4),
        'within_10_points_rate': round(sum(within10) / max(1, n), 4),
        'passed': bool(
            ece <= RELEASE_THRESHOLDS['max_ece_points']
            and (max(bucket_abs_gaps) if bucket_abs_gaps else 0.0) <= RELEASE_THRESHOLDS['max_mce_points']
            and sum(agreement) / max(1, n) >= RELEASE_THRESHOLDS['min_bucket_agreement']
            and sum(within10) / max(1, n) >= RELEASE_THRESHOLDS['min_within_10_points']
        ),
    }

calibration_metrics = {}
for output in sorted({row['output'] for row in calibration_rows}):
    calibration_metrics[output] = calibration_metrics_for([row for row in calibration_rows if row['output'] == output])

slice_calibration = []
for output in sorted({row['output'] for row in calibration_rows}):
    rows = [row for row in calibration_rows if row['output'] == output]
    slice_keys = sorted({key for row in rows for key in row.get('slice', {})})
    for key in slice_keys:
        values = sorted({str(row.get('slice', {}).get(key)) for row in rows})
        for value in values:
            slice_rows = [row for row in rows if str(row.get('slice', {}).get(key)) == value]
            if len(slice_rows) >= 3:
                metrics = calibration_metrics_for(slice_rows)
                slice_calibration.append({'output': output, 'slice_key': key, 'slice_value': value, **metrics})

metric_errors = []
for output, metrics in calibration_metrics.items():
    for key, value in metrics.items():
        if key not in {'passed'} and isinstance(value, (int, float)) and not math.isfinite(value):
            metric_errors.append({'output': output, 'metric': key, 'value': value})
assert not metric_errors, metric_errors
calibration_metrics

{'atsFriendliness.score': {'row_count': 10,
  'ece_points': 4.2,
  'mce_points': 16.0,
  'mae': 4.2,
  'bucket_agreement': 0.9,
  'within_10_points_rate': 0.9,
  'passed': True},
 'jobFitAlignment.score': {'row_count': 200,
  'ece_points': 0.4595,
  'mce_points': 13.8064,
  'mae': 2.2225,
  'bucket_agreement': 0.915,
  'within_10_points_rate': 0.92,
  'passed': True},
 'recommendations[].matchScore': {'row_count': 15,
  'ece_points': 3.0667,
  'mce_points': 8.5,
  'mae': 3.4667,
  'bucket_agreement': 1.0,
  'within_10_points_rate': 0.9333,
  'passed': True}}

## Step 22.4 — Export model card and artifact manifest

### Purpose
Export final model artifacts, feature config, label manifest, schema files, model card, and artifact manifest with SHA-256 hashes.

### Required input
Calibration evidence, prior model metrics, dataset snapshots, label governance reports, selected model artifacts, and schema boundaries.

### Action
Write deployable JSON artifacts under `artifacts/phase_22_calibration_model_card_export/` using relative paths and hash every listed artifact.

### Expected output
Feature config, label manifest, schema file, model card, calibration table, and artifact manifest with hashes.

### Verification
Model card includes git commit, dataset hash, label version, split seed, feature config, metrics, limitations, intended use, and blocked use. Artifact manifest includes inference-required and training-only artifacts with hash and schema version.

In [4]:
selected_model_name = 'high_recall_calibrated_scorer'
selected_model_artifact = ARTIFACTS / 'models/phase_18_jobfit_training_v2/high_recall_calibrated_scorer.joblib'
if not selected_model_artifact.exists():
    selected_model_artifact = ARTIFACTS / 'models/phase_18_jobfit_training_v2/feature_augmented_scorer.joblib'

pairs_path = ARTIFACTS / 'pairs_v2.parquet'
dataset_hash = sha256_file(pairs_path) or snapshot_manifest.get('artifact', {}).get('sha256') or 'unknown'
label_version = label_manifest16.get('label_version') or label_manifest16.get('schema_version') or 'human-validation-v1'

feature_config = {
    'schema_version': 'feature-config-v1',
    'embedding_model': 'intfloat/e5-base-v2',
    'embedding_prefixes': {'profile_cv': 'query:', 'job': 'passage:'},
    'normalized_features': ['skill_overlap', 'requirement_coverage', 'role_match', 'experience_match', 'experience_gap_years', 'language', 'role_family', 'experience_band'],
    'score_outputs': ['jobFitAlignment.score', 'atsFriendliness.score', 'recommendations[].matchScore'],
    'candidate_reranking': {'requires_backend_job_candidates': True, 'max_recommendations': 5, 'static_job_index_allowed': False},
    'split_seed': SPLIT_SEED,
}

export_label_manifest = {
    'schema_version': 'export-label-manifest-v1',
    'label_version': label_version,
    'sources': [
        {'path': 'reports/phase_16_label_manifest.json', 'purpose': 'human validation label governance'},
        {'path': 'reports/phase_19_ats_issue_labels.json', 'purpose': 'ATS benchmark issue taxonomy'},
        {'path': 'reports/phase_21_backend_candidate_reranking.json', 'purpose': 'candidate relevance fixture labels'},
    ],
    'inference_policy': 'labels are evaluation evidence only and must not be used as inference-time features',
}

schema_export = {
    'schema_version': 'model-core-export-schema-v1',
    'outputs': {
        'jobFitAlignment': {'score': 'integer 0-100', 'matchedSkills': 'array[string]', 'missingSkills': 'array[string]', 'summarySignals': 'array[object]', 'confidenceNotes': 'array[string]'},
        'atsFriendliness': {'score': 'integer 0-100', 'detectedIssues': 'array[string]', 'fallback': 'boolean', 'evidence': 'object'},
        'overallImpression': {'score': 'integer 0-100', 'summary': 'string', 'evidenceKeys': 'array[string]', 'confidenceNotes': 'array[string]'},
        'recommendations': {'jobId': 'backend-supplied string', 'matchScore': 'integer 0-100', 'matchLevel': list(('strong', 'good', 'stretch')), 'matchedSkills': 'array[string]', 'missingSkills': 'array[string]', 'rankingSignals': 'array[object]'},
    },
    'forbidden_model_core_fields': ['title', 'companyName', 'visibility', 'availability', 'reason', 'nextStep', 'topActionables', 'sectionReviews'],
}

calibration_export = {
    'schema_version': 'score-calibration-v1',
    'generated_at': GENERATED_AT,
    'buckets': [f'{low}-{high}' for low, high in BUCKETS],
    'tables': calibration_tables,
    'metrics': calibration_metrics,
    'slice_calibration': slice_calibration,
    'thresholds': RELEASE_THRESHOLDS,
}

selected_metrics = phase18_metrics.get('candidate_metrics', {}).get(selected_model_name, {}).get('splits', {})
model_card = {
    'schema_version': 'model-card-v2',
    'model_name': 'bisakerja-model-core-v2',
    'model_version': 'phase-22-export-v1',
    'git_commit': git_commit,
    'git_dirty_at_export': git_dirty,
    'dataset_hash': dataset_hash,
    'dataset_path': rel(pairs_path) if pairs_path.exists() else 'artifacts/pairs_v2.parquet',
    'label_version': label_version,
    'split_seed': SPLIT_SEED,
    'feature_config': feature_config,
    'metrics': {'jobfit_selected_model': selected_metrics, 'ats': phase19_scorer.get('transparent_scorer', {}), 'recommendations': phase21.get('metrics', {}).get('model', {}), 'calibration': calibration_metrics},
    'limitations': [
        'ID and MIXED language slices remain smaller than EN and need more validation labels.',
        'Recommendation calibration uses backend-like fixture labels until larger production relevance labels are available.',
        'Scores are model-core evidence signals, not hiring outcomes or eligibility claims.',
    ],
    'intended_use': 'Model-core CV analysis scoring, ATS evidence scoring, overall impression evidence, and backend-provided candidate reranking.',
    'blocked_use': model_card_template.get('blocked_use', []) + ['Automated rejection, hiring eligibility, salary prediction, or protected-class inference.'],
}

paths = {
    'feature_config': EXPORT_DIR / 'feature_config.json',
    'label_manifest': EXPORT_DIR / 'label_manifest.json',
    'schema': EXPORT_DIR / 'model_core_schema.json',
    'calibration': EXPORT_DIR / 'score_calibration.json',
    'model_card': EXPORT_DIR / 'model_card.json',
}
write_json(paths['feature_config'], feature_config)
write_json(paths['label_manifest'], export_label_manifest)
write_json(paths['schema'], schema_export)
write_json(paths['calibration'], calibration_export)
write_json(paths['model_card'], model_card)

artifact_entries = []
def add_artifact(path: Path, role: str, owner: str, schema_version: str, required_for_inference: bool) -> None:
    artifact_entries.append({
        'path': rel(path) if path.is_absolute() and path.exists() else str(path),
        'role': role,
        'owner': owner,
        'schema_version': schema_version,
        'required_for_inference': required_for_inference,
        'sha256': sha256_file(path if path.is_absolute() else ROOT / path),
    })

for name, path in paths.items():
    add_artifact(path, name, 'model-core', read_json(path, {}).get('schema_version', SCHEMA_VERSION), True)
add_artifact(selected_model_artifact, 'selected_jobfit_model', 'model-core', 'joblib-model-artifact', True)
add_artifact(REPORTS / 'phase_19_ats_scorer_metrics.json', 'ats_scorer_evidence', 'training-eval', 'ats-benchmark-scorer-v1', True)
add_artifact(REPORTS / 'phase_20_overall_impression_signals.json', 'overall_impression_evidence', 'training-eval', 'overall-impression-signals-v1', True)
add_artifact(REPORTS / 'phase_21_backend_candidate_reranking.json', 'candidate_reranking_evidence', 'training-eval', 'backend-candidate-reranking-v1', True)
add_artifact(pairs_path, 'training_pair_dataset', 'training-only', 'pairs-v2', False)
add_artifact(prediction_path, 'prediction_sample', 'training-only', 'prediction-sample-v1', False)
add_artifact(REPORTS / 'phase_18_model_metrics.json', 'jobfit_metrics', 'training-eval', 'jobfit-training-v2', False)
add_artifact(REPORTS / 'phase_16_label_manifest.json', 'human_label_manifest', 'training-eval', 'human-validation-v1', False)

artifact_manifest = {
    'schema_version': 'artifact-manifest-v1',
    'generated_at': GENERATED_AT,
    'git_commit': git_commit,
    'git_dirty_at_export': git_dirty,
    'artifacts': artifact_entries,
}
manifest_path = EXPORT_DIR / 'artifact_manifest.json'
write_json(manifest_path, artifact_manifest)
# Re-read with manifest hash included in final report, but manifest cannot include own stable hash without self-reference.

model_card_required = ['git_commit', 'dataset_hash', 'label_version', 'split_seed', 'feature_config', 'metrics', 'limitations', 'intended_use', 'blocked_use']
model_card_errors = [field for field in model_card_required if field not in model_card or model_card[field] in (None, '', [])]
manifest_errors = [entry for entry in artifact_entries if not entry.get('sha256')]
assert not model_card_errors, model_card_errors
assert not manifest_errors, manifest_errors
len(artifact_entries), rel(manifest_path)

(13, 'artifacts/phase_22_calibration_model_card_export/artifact_manifest.json')

## Step 22.5 — Runtime load checks

### Purpose
Run runtime load checks for exported artifacts from a clean notebook kernel and document environment requirements.

### Required input
Exported artifact manifest, schema, feature config, label manifest, model card, and calibration table.

### Action
Load exported JSON artifacts from disk in an isolated function, validate hashes, relative paths, required fields, and score schema bounds.

### Expected output
`runtime_load_check` and `reports/phase_22_calibration_model_card_export.json`.

### Verification
The exported model-core artifact package loads without hidden notebook variables or local-only absolute paths.

In [5]:
def load_exported_package(export_dir: Path) -> dict[str, Any]:
    required = ['feature_config.json', 'label_manifest.json', 'model_core_schema.json', 'score_calibration.json', 'model_card.json', 'artifact_manifest.json']
    loaded = {}
    for filename in required:
        path = export_dir / filename
        with path.open() as handle:
            loaded[filename] = json.load(handle)
    return loaded

loaded_package = load_exported_package(EXPORT_DIR)
runtime_errors = []
for filename, payload in loaded_package.items():
    if not payload.get('schema_version'):
        runtime_errors.append({'file': filename, 'error': 'missing_schema_version'})

loaded_manifest = loaded_package['artifact_manifest.json']
for entry in loaded_manifest.get('artifacts', []):
    entry_path = ROOT / entry['path']
    if os.path.isabs(entry['path']):
        runtime_errors.append({'path': entry['path'], 'error': 'absolute_path_not_allowed'})
    if entry_path.exists() and sha256_file(entry_path) != entry.get('sha256'):
        runtime_errors.append({'path': entry['path'], 'error': 'sha256_mismatch'})
    if entry.get('required_for_inference') and not entry_path.exists():
        runtime_errors.append({'path': entry['path'], 'error': 'missing_inference_artifact'})

schema_outputs = loaded_package['model_core_schema.json'].get('outputs', {})
for required_output in ['jobFitAlignment', 'atsFriendliness', 'overallImpression', 'recommendations']:
    if required_output not in schema_outputs:
        runtime_errors.append({'output': required_output, 'error': 'missing_output_schema'})

runtime_load_check = {
    'passed': not runtime_errors,
    'errors': runtime_errors,
    'loaded_files': sorted(loaded_package),
    'environment_requirements': {
        'python': '3.10+',
        'core_runtime': ['json', 'hashlib', 'pathlib'],
        'model_runtime': ['joblib-compatible selected scorer artifact', 'sentence-transformers with intfloat/e5-base-v2 for embedding generation'],
        'path_policy': 'all manifest paths are repository-relative',
    },
}

acceptance = {
    'score_meanings_backed_by_calibration_evidence': all(metrics['passed'] for metrics in calibration_metrics.values()) and bool(calibration_tables),
    'model_card_has_required_fields': not model_card_errors,
    'artifact_manifest_hashes_required_and_training_artifacts': not manifest_errors and any(a['required_for_inference'] for a in artifact_entries) and any(not a['required_for_inference'] for a in artifact_entries),
    'exported_model_loads_without_hidden_state': runtime_load_check['passed'],
}

report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete' if all(acceptance.values()) else 'blocked',
    'acceptance': acceptance,
    'calibration_tables': calibration_tables,
    'calibration_metrics': calibration_metrics,
    'slice_calibration': slice_calibration,
    'export_paths': {key: rel(path) for key, path in paths.items()} | {'artifact_manifest': rel(manifest_path)},
    'artifact_manifest_sha256': sha256_file(manifest_path),
    'model_card_summary': {key: model_card[key] for key in model_card_required},
    'runtime_load_check': runtime_load_check,
    'dirty_worktree_at_export': git_dirty,
}
write_json(REPORTS / 'phase_22_calibration_model_card_export.json', report)
assert report['status'] == 'complete', report
report['status'], acceptance

('complete',
 {'score_meanings_backed_by_calibration_evidence': True,
  'model_card_has_required_fields': True,
  'artifact_manifest_hashes_required_and_training_artifacts': True,
  'exported_model_loads_without_hidden_state': True})